# 🛡️ Sudarshana (StegShield) — Account 1 Training Pipeline
### High-Speed T4 GPU Training on Real-World Cyber Threats & Steganography

This notebook trains the Sudarshana Steganalyzer on **NVIDIA T4 GPU**.
Follow the cells in order: **1 → 2 → 3 → 4 → 5 → 6**.

In [ ]:
# ── Cell 1: Check GPU Acceleration ──
import torch
print(f"PyTorch Version: {torch.__version__}")
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Active GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ WARNING: GPU not detected! Go to Runtime -> Change runtime type -> T4 GPU")

In [ ]:
# ── Cell 2: Extract Codebase & Install Requirements ──
import os, sys

# If EDI_Project_Colab_Latest.zip was uploaded to /content or Google Drive:
if os.path.exists('/content/EDI_Project_Colab_Latest.zip'):
    ZIP_PATH = '/content/EDI_Project_Colab_Latest.zip'
elif os.path.exists('/content/drive/MyDrive/EDI_Project_Colab_Latest.zip'):
    ZIP_PATH = '/content/drive/MyDrive/EDI_Project_Colab_Latest.zip'
else:
    from google.colab import files
    print("Please select and upload EDI_Project_Colab_Latest.zip:")
    uploaded = files.upload()
    ZIP_PATH = list(uploaded.keys())[0]

!unzip -q -o "{ZIP_PATH}" -d /content/EDI_Project
%cd /content/EDI_Project
sys.path.insert(0, 'src')
!pip install -q scipy scikit-learn seaborn plotly
print("✅ Workspace ready at /content/EDI_Project")

In [ ]:
# ── Cell 3: Setup Dataset (From Google Drive OR Instant Synthetic Generator) ──
import os
from pathlib import Path

# Optional: Mount Google Drive if your 10,000 dataset is stored there:
try:
    from google.colab import drive
    if not os.path.exists('/content/drive/MyDrive'):
        print('Mounting Google Drive...')
        drive.mount('/content/drive')
except Exception as e:
    print(f'Drive mount notice: {e}')

DRIVE_CLEAN = "/content/drive/MyDrive/EDI_Project_Final/data/clean"
DRIVE_STEGO = "/content/drive/MyDrive/EDI_Project_Final/data/stego"

!mkdir -p data/clean data/stego

if os.path.exists(DRIVE_CLEAN) and os.path.exists(DRIVE_STEGO) and len(list(Path(DRIVE_CLEAN).glob('*.*'))) > 0:
    print("⚡ Found Google Drive dataset! Copying images to fast local SSD...")
    !cp -r "{DRIVE_CLEAN}"/* data/clean/
    !cp -r "{DRIVE_STEGO}"/* data/stego/
else:
    print("ℹ️ Google Drive dataset not found at default path.")
    print("⚡ Generating 300 matched (clean, stego) training pairs with real-world threats...")
    !python scripts/generate_malicious_stego.py --num_images 300 --paired_training_set

clean_count = len(list(Path("data/clean").glob("*.*")))
stego_count = len(list(Path("data/stego").glob("*.*")))
print(f"\n📊 Training Dataset Status:")
print(f"   Clean Covers:   {clean_count} images")
print(f"   Stego Carriers: {stego_count} images")
assert clean_count > 0 and stego_count > 0, 'Error: Dataset is empty!'


In [ ]:
# ── Cell 4: Start GPU Training (Account 1) ──
# Uses Joint Fusion mode with AdamW + Cosine Annealing scheduler & fp16 mixed precision
!python src/training/train.py \
    --mode joint \
    --epochs 30 \
    --lr 1e-4 \
    --batch_size 32 \
    --seed 42

In [ ]:
# ── Cell 5: Evaluate on Real-World Cyber Threats & Test Accuracy ──
import torch
from src.model.fusion_model import MultiBranchSteganalyzer
from src.data_pipeline.dataset import get_dataloaders
from src.utils.checkpoints import extract_state_dict

# 1. Run real-world malware threat evaluation
!python scripts/generate_malicious_stego.py --evaluate

# 2. Calculate held-out test set accuracy
print("\n" + "=" * 60)
print("  Computing Held-out Test Set Accuracy")
print("=" * 60)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
_, _, test_loader = get_dataloaders(
    clean_dir="data/clean",
    stego_dir="data/stego",
    batch_size=32
)

model = MultiBranchSteganalyzer().to(device)
raw = torch.load("weights/fusion_best.pt", map_location=device)
model.load_state_dict(extract_state_dict(raw), strict=False)
model.eval()

correct, total = 0, 0
with torch.no_grad():
    for clean_imgs, stego_imgs in test_loader:
        B = clean_imgs.size(0)
        images = torch.cat([clean_imgs, stego_imgs], dim=0).to(device)
        labels = torch.cat([torch.zeros(B, dtype=torch.long), torch.ones(B, dtype=torch.long)], dim=0).to(device)
        
        logits = model(images)
        preds = logits.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

acc = (correct / total) * 100 if total > 0 else 0
print(f"\n🏆 Final Test Accuracy: {acc:.2f}% ({correct}/{total} images evaluated)")

In [ ]:
# ── Cell 6: Download & Save Trained Weights ──
import os, shutil
from google.colab import files

# 1. Save to Google Drive if connected:
if os.path.exists('/content/drive/MyDrive'):
    shutil.copy2('weights/fusion_best.pt', '/content/drive/MyDrive/fusion_best_account1.pt')
    print("✅ Saved backup to Google Drive: fusion_best_account1.pt")

# 2. Direct browser download to your Mac:
print("📥 Triggering browser download of trained weights...")
files.download('weights/fusion_best.pt')